# IPL Cricket Analysis (2008–2019)

**Author:** Ritinder Kaur  
**Dataset:** 756 matches · 179,077 ball-by-ball deliveries · 12 seasons

---

## Objective

This notebook answers five core questions about IPL performance across 12 seasons:

1. Which teams dominated across the tournament's history?
2. Who are the standout individual performers (batting and bowling)?
3. Does winning the toss actually translate into winning the match?
4. How does run-scoring change across the three phases of an innings?
5. What statistical patterns separate first-innings and second-innings teams?

The analysis uses **cleaned, feature-engineered data** produced by `scripts/clean_data.py`. Key transformations applied:
- Team names standardised across seasons (e.g. *Delhi Daredevils* → *Delhi Capitals*)
- Super-over deliveries removed to avoid score inflation
- Each delivery classified into match phase: **Powerplay** (overs 1–6), **Middle** (7–15), **Death** (16–20)
- Legal delivery flag added for accurate strike rate and economy calculations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

matches    = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')

print(f'Matches:    {matches.shape[0]:,} rows  ×  {matches.shape[1]} columns')
print(f'Deliveries: {deliveries.shape[0]:,} rows  ×  {deliveries.shape[1]} columns')
print(f'Seasons:    {sorted(matches["year"].unique())}')

---
## 1. Dataset Overview

Before any analysis, it's worth understanding the shape and quality of the data.

In [ ]:
matches.head(3)

In [ ]:
# Missing values — matches table
missing = matches.isnull().sum()
missing[missing > 0]

**Data quality notes:**
- `winner` is null for 4 rows — these are tied/no-result matches, excluded from win-rate calculations via `result_clean == 'normal'`
- `player_dismissed` nulls in deliveries are expected — most balls don't produce a dismissal
- `umpire3` (TV umpire) is absent in ~84% of matches — column dropped in analysis where not needed

---
## 2. Team Performance — Who Dominated?

**Question:** Across 12 seasons, which franchises have the best win records?

We look at both raw wins (useful for brand recognition) and win percentage (fairer across teams with different tenures).

In [ ]:
# Raw wins — top 10
team_wins = matches['winner'].value_counts().head(10)
print(team_wins.to_string())

In [ ]:
# Win percentage (teams with ≥ 30 matches played)
total_played = pd.concat([
    matches[['team1']].rename(columns={'team1': 'team'}),
    matches[['team2']].rename(columns={'team2': 'team'})
]).groupby('team').size().reset_index(name='played')

wins = matches.groupby('winner').size().reset_index(name='wins')

team_stats = total_played.merge(wins, left_on='team', right_on='winner', how='left')
team_stats['win_pct'] = (team_stats['wins'] / team_stats['played'] * 100).round(1)
team_stats = team_stats[team_stats['played'] >= 30].sort_values('win_pct', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('IPL Team Performance (2008–2019)', fontsize=14, fontweight='bold')

# Raw wins
axes[0].barh(team_wins.index[::-1], team_wins.values[::-1], color='#1a1a2e')
axes[0].set_title('Total Match Wins (Top 10)')
axes[0].set_xlabel('Wins')
for i, (val, name) in enumerate(zip(team_wins.values[::-1], team_wins.index[::-1])):
    axes[0].text(val + 0.5, i, str(val), va='center', fontsize=9)

# Win %
axes[1].barh(team_stats['team'][::-1], team_stats['win_pct'][::-1], color='#16213e')
axes[1].set_title('Win % (min. 30 matches)')
axes[1].set_xlabel('Win %')
axes[1].axvline(50, color='red', linestyle='--', linewidth=1, label='50% line')
axes[1].legend()
for i, (val, name) in enumerate(zip(team_stats['win_pct'][::-1], team_stats['team'][::-1])):
    axes[1].text(val + 0.3, i, f'{val}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

**Key insight:** Mumbai Indians lead in raw wins (109) and maintain a win rate above 55%, confirming consistent dominance rather than just longevity. Chennai Super Kings (100 wins) are the only other franchise above the 50% threshold when controlling for matches played.

---
## 3. Individual Batting Performance

**Question:** Who are the most prolific run scorers, and how do their strike rates and averages compare?

We filter to batsmen with **20+ matches** to exclude small-sample outliers.

In [ ]:
bat_stats = deliveries.groupby('batsman').agg(
    runs       = ('batsman_runs', 'sum'),
    sixes      = ('is_six',       'sum'),
    fours      = ('is_four',      'sum'),
    dismissals = ('is_wicket',    'sum'),
    balls      = ('is_legal',     'sum'),
    matches    = ('match_id',     'nunique'),
).reset_index()

bat_stats['strike_rate'] = (bat_stats['runs'] / bat_stats['balls'] * 100).round(1)
bat_stats['average']     = (bat_stats['runs'] / bat_stats['dismissals'].replace(0, np.nan)).round(1)
bat_stats = bat_stats[bat_stats['matches'] >= 20].sort_values('runs', ascending=False)

top10 = bat_stats.head(10)
top10[['batsman', 'matches', 'runs', 'average', 'strike_rate', 'sixes', 'fours']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top 10 Batsmen — IPL 2008–2019', fontsize=14, fontweight='bold')

# Runs
axes[0].barh(top10['batsman'][::-1], top10['runs'][::-1], color='#0f3460')
axes[0].set_title('Career IPL Runs')
axes[0].set_xlabel('Total Runs')
for i, (val, name) in enumerate(zip(top10['runs'][::-1], top10['batsman'][::-1])):
    axes[0].text(val + 20, i, f'{val:,}', va='center', fontsize=9)

# Strike rate bubble: avg on x, SR on y, size = runs
scatter = axes[1].scatter(
    top10['average'], top10['strike_rate'],
    s=top10['runs'] / 20,
    color='#e94560', alpha=0.8, edgecolors='black', linewidth=0.5
)
for _, row in top10.iterrows():
    axes[1].annotate(
        row['batsman'].split()[-1],
        (row['average'], row['strike_rate']),
        textcoords='offset points', xytext=(5, 3), fontsize=8
    )
axes[1].set_xlabel('Batting Average')
axes[1].set_ylabel('Strike Rate')
axes[1].set_title('Average vs Strike Rate\n(bubble size = total runs)')

plt.tight_layout()
plt.show()

**Key insight:** Virat Kohli leads with 5,434 runs, but the scatter plot reveals the real story — players like AB de Villiers combine a high average with an elite strike rate, making them the most *efficient* batsmen despite lower totals. Kohli's consistency (average > 30) underpins his volume.

---
## 4. Individual Bowling Performance

**Question:** Which bowlers took the most wickets, and how efficiently did they concede runs?

Filter: bowlers with **15+ matches and 20+ wickets**.

In [ ]:
bowl_stats = deliveries.groupby('bowler').agg(
    wickets    = ('is_wicket',  'sum'),
    runs_given = ('total_runs', 'sum'),
    legal_balls= ('is_legal',   'sum'),
    matches    = ('match_id',   'nunique'),
).reset_index()

bowl_stats['economy'] = (bowl_stats['runs_given'] / bowl_stats['legal_balls'] * 6).round(2)
bowl_stats['average'] = (bowl_stats['runs_given'] / bowl_stats['wickets'].replace(0, np.nan)).round(1)
bowl_stats = bowl_stats[
    (bowl_stats['matches'] >= 15) & (bowl_stats['wickets'] >= 20)
].sort_values('wickets', ascending=False)

top10_bowl = bowl_stats.head(10)
top10_bowl[['bowler', 'matches', 'wickets', 'economy', 'average']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Top 10 Bowlers — IPL 2008–2019', fontsize=14, fontweight='bold')

# Wickets
axes[0].barh(top10_bowl['bowler'][::-1], top10_bowl['wickets'][::-1], color='#2d6a4f')
axes[0].set_title('Career IPL Wickets')
axes[0].set_xlabel('Wickets')
for i, val in enumerate(top10_bowl['wickets'][::-1]):
    axes[0].text(val + 0.5, i, str(int(val)), va='center', fontsize=9)

# Economy vs Average scatter
axes[1].scatter(
    top10_bowl['economy'], top10_bowl['average'],
    s=top10_bowl['wickets'] * 3,
    color='#40916c', alpha=0.8, edgecolors='black', linewidth=0.5
)
for _, row in top10_bowl.iterrows():
    axes[1].annotate(
        row['bowler'].split()[-1],
        (row['economy'], row['average']),
        textcoords='offset points', xytext=(5, 3), fontsize=8
    )
axes[1].set_xlabel('Economy Rate')
axes[1].set_ylabel('Bowling Average (lower = better)')
axes[1].set_title('Economy vs Bowling Average\n(bubble size = wickets)')

plt.tight_layout()
plt.show()

**Key insight:** Lasith Malinga leads in wickets (188) and maintains an economy below 7 — rare combination of volume *and* efficiency. The scatter plot shows that spinners (Harbhajan Singh, R Ashwin, SP Narine) tend to cluster at lower economy rates, reflecting their role in restricting middle-over run flow.

---
## 5. Toss Impact — Does the Coin Flip Matter?

**Question:** Do teams that win the toss win more matches? And does the choice to bat or field make a difference?

Conventional cricket wisdom says fielding first (chasing) is advantageous in T20 — let's test this with the data.

In [ ]:
# Overall toss advantage
overall_toss_adv = matches['toss_won_match'].mean() * 100
print(f'Toss winner won the match: {overall_toss_adv:.1f}% of the time')
print(f'(Random chance baseline: 50.0%)')

# By decision
toss_by_decision = matches.groupby('toss_decision')['toss_won_match'].mean().reset_index()
toss_by_decision['win_pct'] = (toss_by_decision['toss_won_match'] * 100).round(1)
print('\nWin % by toss decision:')
print(toss_by_decision[['toss_decision', 'win_pct']].to_string(index=False))

# Bat-first win rate overall
bat_first_win = matches['bat_first_won'].mean() * 100
print(f'\nBat-first team won: {bat_first_win:.1f}% of matches')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Toss Impact Analysis — IPL 2008–2019', fontsize=14, fontweight='bold')

# Win % by decision
colors = ['#264653', '#2a9d8f']
bars = axes[0].bar(
    toss_by_decision['toss_decision'],
    toss_by_decision['win_pct'],
    color=colors, width=0.4
)
axes[0].axhline(50, color='red', linestyle='--', linewidth=1.2, label='50% baseline')
axes[0].set_ylim(40, 65)
axes[0].set_title('Toss Winner Win % by Decision')
axes[0].set_ylabel('Win %')
axes[0].legend()
for bar, val in zip(bars, toss_by_decision['win_pct']):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 0.3, f'{val}%',
                 ha='center', fontsize=11, fontweight='bold')

# Bat-first vs chase win rate
categories = ['Bat First', 'Chase']
values = [bat_first_win, 100 - bat_first_win]
axes[1].bar(categories, values, color=['#e76f51', '#f4a261'], width=0.4)
axes[1].axhline(50, color='red', linestyle='--', linewidth=1.2, label='50% baseline')
axes[1].set_ylim(40, 65)
axes[1].set_title('Bat-First vs Chase Win Rate')
axes[1].set_ylabel('Win %')
axes[1].legend()
for i, val in enumerate(values):
    axes[1].text(i, val + 0.3, f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

**Key insight:** Teams that choose to field first after winning the toss win at a notably higher rate than those who bat first — supporting the T20 conventional wisdom that chasing is advantageous. However, the chi-square test in `scripts/statistical_analysis.py` confirms whether this association is statistically significant or could be due to chance.

---
## 6. Phase Analysis — When Are Runs Scored?

**Question:** How does the average run rate change across Powerplay, Middle overs, and Death overs?

Understanding run-scoring by phase reveals when bowlers are most under pressure and how teams build innings.

In [ ]:
phase_runs = deliveries.groupby(['match_id', 'inning', 'phase'])['total_runs'].sum().reset_index()
phase_avg  = phase_runs.groupby('phase')['total_runs'].agg(['mean', 'median', 'std']).round(2)
phase_avg  = phase_avg.reindex(['Powerplay', 'Middle', 'Death'])
print('Average runs per phase per innings:\n')
print(phase_avg.rename(columns={'mean': 'Mean', 'median': 'Median', 'std': 'Std Dev'}).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Run Scoring by Match Phase — IPL 2008–2019', fontsize=14, fontweight='bold')

phase_order = ['Powerplay', 'Middle', 'Death']
phase_colors = ['#457b9d', '#1d3557', '#e63946']

means = [phase_avg.loc[p, 'mean'] for p in phase_order]

axes[0].bar(phase_order, means, color=phase_colors, width=0.5)
axes[0].set_title('Average Runs per Phase')
axes[0].set_ylabel('Average Runs')
for i, val in enumerate(means):
    axes[0].text(i, val + 0.2, f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')

# Distribution as violin
phase_long = phase_runs[phase_runs['phase'].isin(phase_order)].copy()
phase_long['phase'] = pd.Categorical(phase_long['phase'], categories=phase_order, ordered=True)
sns.violinplot(
    data=phase_long, x='phase', y='total_runs',
    palette=dict(zip(phase_order, phase_colors)),
    ax=axes[1], inner='quartile'
)
axes[1].set_title('Run Distribution by Phase')
axes[1].set_ylabel('Runs per Phase per Innings')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

**Key insight:** Death overs produce the highest average runs per over — teams attack in the final 5 overs to maximise totals. The wide violin distribution in the Death phase shows high variability: a team can score anywhere from 10 to 80+ runs depending on wickets in hand and match situation. Middle overs are the most consistent phase, with teams accumulating steadily rather than going all-out.

---
## 7. Dismissal Types

**Question:** How do batsmen get out in T20 cricket? Understanding dismissal patterns helps teams plan bowling strategies.

In [ ]:
dismiss_counts = (
    deliveries[deliveries['dismissal_kind'].notna()]
    ['dismissal_kind']
    .value_counts()
)
print(dismiss_counts.to_string())

In [ ]:
top6 = dismiss_counts.head(6)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dismissal Type Analysis — IPL 2008–2019', fontsize=14, fontweight='bold')

# Pie
palette = ['#264653', '#2a9d8f', '#e9c46a', '#f4a261', '#e76f51', '#6d6875']
axes[0].pie(top6, labels=top6.index, autopct='%1.1f%%', colors=palette, startangle=90)
axes[0].set_title('Top 6 Dismissal Types')

# Bar (all types)
axes[1].barh(dismiss_counts.index[::-1], dismiss_counts.values[::-1], color='#264653')
axes[1].set_title('All Dismissal Types')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

**Key insight:** **Caught** dismissals dominate, accounting for over 50% of all wickets — reflecting T20's aggressive batting style, where batsmen attempt aerial shots that find fielders. **Bowled** is the second most common mode, while **LBW** is less frequent than in Test cricket due to the higher proportion of short-pitched and wide deliveries.

---
## 8. Summary of Findings

| # | Question | Finding |
|---|---|---|
| 1 | Which team dominated? | **Mumbai Indians** — 109 wins, >55% win rate |
| 2 | Top run scorer? | **Virat Kohli** — 5,434 runs with avg > 30 |
| 3 | Top wicket taker? | **Lasith Malinga** — 188 wickets at sub-7 economy |
| 4 | Does the toss matter? | Teams choosing to field win at a higher rate; toss-winners overall win ~51% — barely above chance |
| 5 | When are runs scored? | Death overs yield the most runs per over; middle overs are the most consistent phase |
| 6 | How do batsmen get out? | Caught dismissals account for >50% of all wickets |

---

## Next Steps (see `scripts/` folder)

- **`statistical_analysis.py`** — Chi-square test on toss vs outcome; bootstrap CI on innings score; Mann-Whitney U for innings comparison
- **`run_sql_queries.py`** — SQL window functions for per-season win share and strike rate leaderboards
- **`win_prediction.py`** — Logistic Regression predicting 2nd innings outcomes at the 10-over mark
- **`eda_analysis.py`** — Exports `assets/ipl_dashboard.png` with the 4-panel summary dashboard